# Heteroskedastic DGP: separately tuned no-alignment vs quantile-alignment methods

This notebook compares two usable conditional generators rather than enforcing a strict
ablation.  The heteroskedastic DGP has a Z-dependent conditional variance, so symmetric
conditional quantiles are a more relevant auxiliary target than a conditional mean or
median alone.

The workflow deliberately separates tuning from final evaluation:

1. compare several quantile-penalty weights under H0 using common random seeds;
2. select the weight whose rejection rates are closest to 0.10 and 0.05, with an extra
   penalty for over-rejection;
3. rerun a fresh H0 calibration with independent seeds;
4. use those matching H0 p-values for size-adjusted power.

Final experiment scale is unchanged: n=400, 150 final H0 replications per method, and
150 H1 replications per dependence-strength value.  Keep this notebook beside the
modified ci_test.py that implements size_adjusted_cutoffs and null_pvalues.

## 0. Setup and multi-GPU wrapper

GPU 0 is skipped by default.  Change GPU_IDS if the available devices differ.
The wrapper adds seed_offset so lambda tuning and final H0 calibration can use independent
Monte-Carlo samples without modifying ci_test.py.

In [ ]:
%pip install matplotlib pandas torch joblib

In [ ]:
from copy import deepcopy
import importlib
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from joblib import Parallel, delayed

import ci_test as C
C = importlib.reload(C)

assert hasattr(C, 'size_adjusted_cutoffs'), (
    'The imported ci_test.py is old: size_adjusted_cutoffs is missing.'
)
assert 'null_pvalues' in inspect.signature(C.run_experiment).parameters, (
    'The imported ci_test.py is old: run_experiment has no null_pvalues argument.'
)

GPU_IDS = [1, 2, 3]
_ORIG_RUN_EXPERIMENT = C.run_experiment


def _split_seeds(n_rep, n_workers, seed_offset=0):
    seeds = list(range(int(seed_offset), int(seed_offset) + int(n_rep)))
    if n_workers <= 0:
        return [seeds]
    base, rem = divmod(len(seeds), n_workers)
    chunks, start = [], 0
    for i in range(n_workers):
        size = base + (1 if i < rem else 0)
        chunks.append(seeds[start:start + size])
        start += size
    return chunks


def _multi_gpu_chunk_worker(gpu_id, seeds, params):
    import os
    os.environ['CUDA_VISIBLE_DEVICES'] = str(int(gpu_id))

    import numpy as np
    import ci_test as ci_local

    pvalues = []
    for seed in seeds:
        pvalues.append(
            ci_local._one_replicate(
                int(seed),
                params['n'],
                params['hypothesis'],
                params['config'],
                params['oracle'],
                params['data_kwargs'],
                True,
                params['dgp'],
            )
        )
    return list(seeds), np.asarray(pvalues, dtype=float)


def _finish_result(pvalues, hypothesis, levels, null_pvalues, verbose, tag):
    pvalues = np.asarray(pvalues, dtype=float)
    is_h1 = hypothesis == 'H1'

    if is_h1:
        if null_pvalues is None:
            raise ValueError('H1 size-adjusted power requires matching H0 p-values.')
        cutoffs = C.size_adjusted_cutoffs(null_pvalues, levels=levels)
        cutoffs = {float(k): float(v) for k, v in dict(cutoffs).items()}
        adjusted_power = {
            float(level): float(np.mean(pvalues <= cutoffs[float(level)]))
            for level in levels
        }
        if verbose:
            print(
                tag + ' SIZE-ADJUSTED ' + '  '.join(
                    f'power@{level:.2f}={adjusted_power[float(level)]:.3f} '
                    f'(cutoff={cutoffs[float(level)]:.4f})'
                    for level in levels
                )
            )
        return {
            'rejection': adjusted_power,
            'size_adjusted_power': adjusted_power,
            'pvalues': pvalues,
            'cutoffs': cutoffs,
        }

    rejection = {
        float(level): float(np.mean(pvalues < float(level)))
        for level in levels
    }
    if verbose:
        print(
            tag + ' ' + '  '.join(
                f'rej@{level:.2f}={rejection[float(level)]:.3f}'
                for level in levels
            )
        )
    return {'rejection': rejection, 'pvalues': pvalues}


def run_experiment(
    n=400,
    hypothesis='H0',
    n_rep=200,
    config=None,
    oracle=False,
    levels=(0.10, 0.05),
    data_kwargs=None,
    dgp='skew',
    n_jobs=1,
    prefer_gpu=True,
    verbose=True,
    null_pvalues=None,
    gpu_ids=None,
    seed_offset=0,
    **kwargs,
):
    data_kwargs = dict(data_kwargs or {})
    levels = tuple(float(level) for level in levels)
    ids = [int(gpu) for gpu in (GPU_IDS if gpu_ids is None else gpu_ids)]
    use_multi = bool(prefer_gpu) and torch.cuda.is_available() and len(ids) >= 1

    gtag = C.get_dgp(dgp).name
    mtag = 'ORACLE' if oracle else f"depth={(config or {}).get('depth', C.DEFAULT_CONFIG['depth'])}"
    tag = f'[{hypothesis} dgp={gtag} {mtag} n={n} reps={n_rep}]'

    if not use_multi and int(seed_offset) == 0:
        return _ORIG_RUN_EXPERIMENT(
            n=n,
            hypothesis=hypothesis,
            n_rep=n_rep,
            config=config,
            oracle=oracle,
            levels=levels,
            data_kwargs=data_kwargs,
            dgp=dgp,
            n_jobs=n_jobs,
            prefer_gpu=prefer_gpu,
            verbose=verbose,
            null_pvalues=null_pvalues,
            **kwargs,
        )

    seeds = list(range(int(seed_offset), int(seed_offset) + int(n_rep)))

    if not use_multi:
        if n_jobs == 1:
            pvalues = [
                C._one_replicate(
                    seed, n, hypothesis, config, oracle, data_kwargs, False, dgp
                )
                for seed in seeds
            ]
        else:
            pvalues = Parallel(n_jobs=n_jobs)(
                delayed(C._one_replicate)(
                    seed, n, hypothesis, config, oracle, data_kwargs, False, dgp
                )
                for seed in seeds
            )
        return _finish_result(
            pvalues, hypothesis, levels, null_pvalues, verbose, tag
        )

    chunks = _split_seeds(n_rep, len(ids), seed_offset=seed_offset)
    params = {
        'n': n,
        'hypothesis': hypothesis,
        'config': config,
        'oracle': oracle,
        'data_kwargs': data_kwargs,
        'dgp': dgp,
    }
    jobs = [(gpu, chunk) for gpu, chunk in zip(ids, chunks) if chunk]

    if verbose:
        print(
            f'[info] multi-GPU devices={[gpu for gpu, _ in jobs]} | '
            f'reps={n_rep} | seed_offset={seed_offset} | '
            f'split={[len(chunk) for _, chunk in jobs]}'
        )

    parts = Parallel(n_jobs=len(jobs), backend='loky', verbose=0)(
        delayed(_multi_gpu_chunk_worker)(gpu, chunk, params)
        for gpu, chunk in jobs
    )

    seed_to_pvalue = {}
    for seed_list, values in parts:
        for seed, pvalue in zip(seed_list, values):
            seed_to_pvalue[int(seed)] = float(pvalue)
    pvalues = np.asarray([seed_to_pvalue[seed] for seed in seeds], dtype=float)

    return _finish_result(
        pvalues, hypothesis, levels, null_pvalues, verbose, tag
    )


C.run_experiment = run_experiment

print('ci_test loaded from:', Path(C.__file__).resolve())
print('Size-adjusted API check: PASSED')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Multi-GPU wrapper enabled. GPU_IDS =', GPU_IDS)

## 1. Global experiment settings

The final sample size and replication counts match the previous heteroskedastic notebook.
Lambda tuning uses the same number of H0 replications as the final H0 experiment.

In [ ]:
RUN_PROFILE = 'final'      # 'quick' or 'final'

if RUN_PROFILE == 'quick':
    N_REP_ORACLE = 20
    N_REP_LAMBDA_TUNE = 40
    N_REP_H0 = 40
    N_REP_H1 = 40
else:
    N_REP_ORACLE = 100
    N_REP_LAMBDA_TUNE = 150
    N_REP_H0 = 150
    N_REP_H1 = 150

N = 400
LEVELS = (0.10, 0.05)
DGP_NAME = 'heteroskedastic'
N_JOBS = -1
PREFER_GPU = True
BASE_DATA_KWARGS = {}

ALPHA_GRID = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

# Common random numbers within each stage; independent samples across stages.
LAMBDA_TUNE_SEED_OFFSET = 0
FINAL_H0_SEED_OFFSET = 100_000
POWER_SEED_OFFSET = 200_000

print({
    'profile': RUN_PROFILE,
    'n': N,
    'levels': LEVELS,
    'lambda-tuning H0 reps per candidate': N_REP_LAMBDA_TUNE,
    'final H0 reps per method': N_REP_H0,
    'H1 reps per alpha_x and method': N_REP_H1,
    'DGP': DGP_NAME,
})

## 2. Separately tuned generator configurations

The no-alignment generator receives longer pure-MMD training.  The quantile generator
uses a smaller learning rate than the unsuccessful earlier run, a strong MMD component,
and symmetric quantiles to target conditional spread.  Lambda is filled in after H0 tuning.

In [ ]:
# Method 1: pure conditional MMD, no alignment.
CONFIG_NO_ALIGNMENT = dict(C.DEFAULT_CONFIG)
CONFIG_NO_ALIGNMENT.update(
    depth=2,
    width=1024,
    noise_dim=5,
    dropout=0.0,
    lr=5e-4,
    epochs=900,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=50,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,
    early_stop=True,
    min_epochs=120,
    patience=100,
    min_delta=1e-5,
    lr_scheduler=True,
    lr_factor=0.4,
    lr_patience=15,
    min_lr_frac=0.02,
    align_mode='none',
    lambda_align=0.0,
    taus=(0.1, 0.5, 0.9),
    align_samples=64,
    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv='gaussian',
    standardize=True,
)

# Method 2 base: conditional MMD plus spread-aware quantile alignment.
CONFIG_QUANTILE_BASE = dict(C.DEFAULT_CONFIG)
CONFIG_QUANTILE_BASE.update(
    depth=2,
    width=1024,
    noise_dim=5,
    dropout=0.0,
    lr=5e-4,
    epochs=800,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=50,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,
    early_stop=True,
    min_epochs=100,
    patience=80,
    min_delta=1e-5,
    lr_scheduler=True,
    lr_factor=0.4,
    lr_patience=15,
    min_lr_frac=0.02,
    align_mode='quantile',
    lambda_align=0.10,       # temporary fallback; replaced after tuning
    taus=(0.1, 0.5, 0.9),
    align_samples=128,
    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv='gaussian',
    standardize=True,
)

# Include 0.5 to verify the earlier aggressive choice, but prefer smaller values on ties.
LAMBDA_GRID = [0.05, 0.10, 0.20, 0.30, 0.50]
FALLBACK_LAMBDA = 0.10

print('Lambda candidates:', LAMBDA_GRID)

## 3. Tune lambda under H0

Each candidate uses the same Monte-Carlo seeds.  The selection score is the sum of
squared, standard-error-scaled deviations from 0.10 and 0.05, plus an extra penalty for
over-rejection.  Candidates inside both approximate 95% binomial fluctuation bands are
preferred.  Power is not used for tuning.

In [ ]:
def binomial_band(level, replications, z_value=1.96):
    standard_error = np.sqrt(level * (1.0 - level) / replications)
    return (
        max(0.0, level - z_value * standard_error),
        min(1.0, level + z_value * standard_error),
    )


def size_tuning_score(rejection_010, rejection_005, replications):
    se_010 = np.sqrt(0.10 * 0.90 / replications)
    se_005 = np.sqrt(0.05 * 0.95 / replications)

    two_sided = (
        ((rejection_010 - 0.10) / se_010) ** 2
        + ((rejection_005 - 0.05) / se_005) ** 2
    )
    over_rejection = (
        (max(0.0, rejection_010 - 0.10) / se_010) ** 2
        + (max(0.0, rejection_005 - 0.05) / se_005) ** 2
    )
    return float(two_sided + 0.5 * over_rejection)


LAMBDA_TUNING_RESULTS = {}
lambda_rows = []

band_010 = binomial_band(0.10, N_REP_LAMBDA_TUNE)
band_005 = binomial_band(0.05, N_REP_LAMBDA_TUNE)

for lambda_value in LAMBDA_GRID:
    config = deepcopy(CONFIG_QUANTILE_BASE)
    config['lambda_align'] = float(lambda_value)

    print()
    print('=' * 88)
    print(f'Lambda tuning under H0: lambda_align={lambda_value:.2f}')
    print('=' * 88)

    result = C.run_experiment(
        n=N,
        hypothesis='H0',
        n_rep=N_REP_LAMBDA_TUNE,
        config=config,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs=BASE_DATA_KWARGS,
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=True,
        seed_offset=LAMBDA_TUNE_SEED_OFFSET,
    )
    LAMBDA_TUNING_RESULTS[float(lambda_value)] = result

    rejection_010 = result['rejection'][0.10]
    rejection_005 = result['rejection'][0.05]
    within_both = (
        band_010[0] <= rejection_010 <= band_010[1]
        and band_005[0] <= rejection_005 <= band_005[1]
    )

    lambda_rows.append({
        'lambda_align': float(lambda_value),
        'raw_H0_rejection_0.10': rejection_010,
        'raw_H0_rejection_0.05': rejection_005,
        'normal_band_0.10_low': band_010[0],
        'normal_band_0.10_high': band_010[1],
        'normal_band_0.05_low': band_005[0],
        'normal_band_0.05_high': band_005[1],
        'within_both_normal_bands': bool(within_both),
        'selection_score': size_tuning_score(
            rejection_010, rejection_005, N_REP_LAMBDA_TUNE
        ),
        'H0_replications': len(result['pvalues']),
    })

LAMBDA_TUNING_TABLE = pd.DataFrame(lambda_rows).sort_values('lambda_align')
display(LAMBDA_TUNING_TABLE)

In [ ]:
# Prefer candidates within both fluctuation bands.  On an exact score tie, choose
# the smaller lambda to reduce unnecessary regularization.
eligible = LAMBDA_TUNING_TABLE[
    LAMBDA_TUNING_TABLE['within_both_normal_bands']
].copy()
if eligible.empty:
    eligible = LAMBDA_TUNING_TABLE.copy()

selected_row = eligible.sort_values(
    ['selection_score', 'lambda_align'],
    ascending=[True, True],
).iloc[0]

SELECTED_LAMBDA = float(selected_row['lambda_align'])

print('Selected lambda_align =', SELECTED_LAMBDA)
print('Selection row:')
display(selected_row.to_frame().T)

if SELECTED_LAMBDA >= 0.5:
    print(
        'Warning: lambda=0.5 was selected. Inspect training stability and the full '
        'H0 table before proceeding because the earlier aggressive configuration '
        'over-rejected at level 0.10.'
    )

In [ ]:
plt.figure(figsize=(8.0, 5.0))
plt.plot(
    LAMBDA_TUNING_TABLE['lambda_align'],
    LAMBDA_TUNING_TABLE['raw_H0_rejection_0.10'],
    marker='o',
    label='Raw H0 rejection at 0.10',
)
plt.plot(
    LAMBDA_TUNING_TABLE['lambda_align'],
    LAMBDA_TUNING_TABLE['raw_H0_rejection_0.05'],
    marker='o',
    label='Raw H0 rejection at 0.05',
)
plt.axhline(0.10, color='C0', linestyle='--', alpha=0.6)
plt.axhline(0.05, color='C1', linestyle='--', alpha=0.6)
plt.axvline(SELECTED_LAMBDA, color='black', linestyle=':', label='Selected lambda')
plt.xlabel('Quantile alignment weight lambda')
plt.ylabel('Raw H0 rejection rate')
plt.title('Lambda tuning for the heteroskedastic DGP')
plt.ylim(0.0, max(0.22, 1.15 * LAMBDA_TUNING_TABLE[
    ['raw_H0_rejection_0.10', 'raw_H0_rejection_0.05']
].to_numpy().max()))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Build the two final methods

The selected quantile weight is inserted into the final quantile configuration.  The
configuration-difference table documents every separately tuned parameter difference.

In [ ]:
CONFIG_QUANTILE_TUNED = deepcopy(CONFIG_QUANTILE_BASE)
CONFIG_QUANTILE_TUNED['lambda_align'] = SELECTED_LAMBDA

METHODS = {
    'no_alignment': CONFIG_NO_ALIGNMENT,
    'quantile_alignment': CONFIG_QUANTILE_TUNED,
}
METHOD_LABELS = {
    'no_alignment': 'No alignment (lambda=0)',
    'quantile_alignment': f'Quantile alignment (lambda={SELECTED_LAMBDA:g})',
}


def config_difference_table(config_a, config_b):
    rows = []
    for key in sorted(set(config_a) | set(config_b)):
        value_a = config_a.get(key, '<missing>')
        value_b = config_b.get(key, '<missing>')
        if value_a != value_b:
            rows.append({
                'parameter': key,
                'no_alignment': value_a,
                'quantile_alignment': value_b,
            })
    return pd.DataFrame(rows)


CONFIG_DIFFERENCES = config_difference_table(
    METHODS['no_alignment'], METHODS['quantile_alignment']
)
display(CONFIG_DIFFERENCES)

## 5. Oracle H0 sanity check

Oracle size checks the statistic and bootstrap independently of generator training.

In [ ]:
ORACLE_RESULT = C.run_experiment(
    n=N,
    hypothesis='H0',
    n_rep=N_REP_ORACLE,
    config=METHODS['no_alignment'],
    dgp=DGP_NAME,
    oracle=True,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
    seed_offset=FINAL_H0_SEED_OFFSET + 50_000,
)
ORACLE_RESULT

## 6. Fresh final H0 calibration

These seeds do not overlap with lambda tuning.  Each final method receives its own H0
p-values, which must be used for its later size-adjusted power calculation.

In [ ]:
H0_RESULTS = {}

for method_id, config in METHODS.items():
    print()
    print('=' * 88)
    print('Final H0 calibration:', METHOD_LABELS[method_id])
    print('=' * 88)

    H0_RESULTS[method_id] = C.run_experiment(
        n=N,
        hypothesis='H0',
        n_rep=N_REP_H0,
        config=config,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs=BASE_DATA_KWARGS,
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=True,
        seed_offset=FINAL_H0_SEED_OFFSET,
    )

In [ ]:
calibration_rows = []
ADJUSTED_CUTOFFS = {}

for method_id, h0_result in H0_RESULTS.items():
    p0 = np.asarray(h0_result['pvalues'])
    cutoffs = C.size_adjusted_cutoffs(p0, levels=LEVELS)
    ADJUSTED_CUTOFFS[method_id] = cutoffs

    for level in LEVELS:
        cutoff = cutoffs[float(level)]
        normal_low, normal_high = binomial_band(float(level), N_REP_H0)
        raw_rejection = float(np.mean(p0 < float(level)))
        calibration_rows.append({
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'nominal_level': float(level),
            'raw_H0_rejection': raw_rejection,
            'normal_band_low': normal_low,
            'normal_band_high': normal_high,
            'raw_H0_within_normal_band': bool(
                normal_low <= raw_rejection <= normal_high
            ),
            'adjusted_cutoff': float(cutoff),
            'H0_rejection_at_adjusted_cutoff': float(np.mean(p0 <= cutoff)),
            'H0_replications': len(p0),
        })

CALIBRATION_TABLE = pd.DataFrame(calibration_rows)
display(CALIBRATION_TABLE)

FINAL_H0_VALID = bool(CALIBRATION_TABLE['raw_H0_within_normal_band'].all())
print('Both methods pass raw H0 fluctuation checks:', FINAL_H0_VALID)
if not FINAL_H0_VALID:
    print(
        'Stop before power: at least one method is outside its normal H0 band. '
        'Inspect generator convergence and rerun a fresh final H0 calibration.'
    )

In [ ]:
H0_FILE = Path('h0_calibration_heteroskedastic_quantile_v2.npz')
TUNING_FILE = Path('lambda_tuning_heteroskedastic_quantile_v2.npz')

np.savez_compressed(
    H0_FILE,
    **{
        method_id: np.asarray(result['pvalues'])
        for method_id, result in H0_RESULTS.items()
    },
)
np.savez_compressed(
    TUNING_FILE,
    **{
        f"lambda_{str(lambda_value).replace('.', 'p')}": np.asarray(result['pvalues'])
        for lambda_value, result in LAMBDA_TUNING_RESULTS.items()
    },
)

CALIBRATION_TABLE.to_csv(
    'h0_calibration_summary_heteroskedastic_quantile_v2.csv', index=False
)
LAMBDA_TUNING_TABLE.to_csv(
    'lambda_tuning_summary_heteroskedastic_quantile_v2.csv', index=False
)
pd.DataFrame([{'selected_lambda': SELECTED_LAMBDA}]).to_csv(
    'selected_lambda_heteroskedastic_quantile_v2.csv', index=False
)

print('Saved final H0 p-values to:', H0_FILE.resolve())
print('Saved lambda-tuning p-values to:', TUNING_FILE.resolve())

### 6.1 Optional reload after a kernel restart

Run the following cell only when the saved tuning selection and final H0 p-values already
exist.  It reconstructs the two final methods without rerunning lambda tuning or H0.

In [ ]:
# Uncomment this cell only after a kernel restart.
# selection = pd.read_csv('selected_lambda_heteroskedastic_quantile_v2.csv')
# SELECTED_LAMBDA = float(selection.loc[0, 'selected_lambda'])
# CONFIG_QUANTILE_TUNED = deepcopy(CONFIG_QUANTILE_BASE)
# CONFIG_QUANTILE_TUNED['lambda_align'] = SELECTED_LAMBDA
# METHODS = {
#     'no_alignment': CONFIG_NO_ALIGNMENT,
#     'quantile_alignment': CONFIG_QUANTILE_TUNED,
# }
# METHOD_LABELS = {
#     'no_alignment': 'No alignment (lambda=0)',
#     'quantile_alignment': f'Quantile alignment (lambda={SELECTED_LAMBDA:g})',
# }
# loaded = np.load('h0_calibration_heteroskedastic_quantile_v2.npz')
# H0_RESULTS = {}
# for method_id in METHODS:
#     p0 = np.asarray(loaded[method_id], dtype=float)
#     H0_RESULTS[method_id] = {
#         'pvalues': p0,
#         'rejection': {
#             float(level): float(np.mean(p0 < float(level)))
#             for level in LEVELS
#         },
#     }
# print('Reloaded selected lambda:', SELECTED_LAMBDA)
# print('Reloaded final H0 methods:', list(H0_RESULTS))

## 7. H1 size-adjusted power

The same power seeds are used for both methods at each alpha_x, while each method uses
its own matching final H0 p-values and adjusted cutoffs.

In [ ]:
POWER_ROWS = []
H1_RESULTS = {method_id: {} for method_id in METHODS}

assert FINAL_H0_VALID, (
    'Power is blocked because at least one final raw H0 rejection rate is outside '
    'its approximate 95% Monte-Carlo fluctuation band.'
)

for alpha_x in ALPHA_GRID:
    print()
    print('-' * 88)
    print(f'alpha_x = {alpha_x:.2f}')
    print('-' * 88)

    for method_id, config in METHODS.items():
        h1_data_kwargs = dict(BASE_DATA_KWARGS)
        h1_data_kwargs['alpha_x'] = float(alpha_x)

        result = C.run_experiment(
            n=N,
            hypothesis='H1',
            n_rep=N_REP_H1,
            config=config,
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs=h1_data_kwargs,
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=False,
            null_pvalues=H0_RESULTS[method_id],
            seed_offset=POWER_SEED_OFFSET,
        )

        H1_RESULTS[method_id][float(alpha_x)] = result
        row = {
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'selected_lambda': 0.0 if method_id == 'no_alignment' else SELECTED_LAMBDA,
            'alpha_x': float(alpha_x),
            'size_adjusted_power_0.10': result['size_adjusted_power'][0.10],
            'size_adjusted_power_0.05': result['size_adjusted_power'][0.05],
            'cutoff_0.10': result['cutoffs'][0.10],
            'cutoff_0.05': result['cutoffs'][0.05],
            'H1_replications': len(result['pvalues']),
        }
        POWER_ROWS.append(row)

        print(
            f"{METHOD_LABELS[method_id]:40s} "
            f"adjusted power@0.10={row['size_adjusted_power_0.10']:.3f}  "
            f"adjusted power@0.05={row['size_adjusted_power_0.05']:.3f}"
        )

POWER_TABLE = pd.DataFrame(POWER_ROWS)
display(POWER_TABLE)

## 8. Power comparison tables and curves

In [ ]:
POWER_COMPARISON_005 = POWER_TABLE.pivot(
    index='alpha_x', columns='method', values='size_adjusted_power_0.05'
).sort_index()
POWER_COMPARISON_010 = POWER_TABLE.pivot(
    index='alpha_x', columns='method', values='size_adjusted_power_0.10'
).sort_index()

print('Size-adjusted power at empirical size 0.05')
display(POWER_COMPARISON_005)
print('Size-adjusted power at empirical size 0.10')
display(POWER_COMPARISON_010)

In [ ]:
for level, value_column in [
    (0.05, 'size_adjusted_power_0.05'),
    (0.10, 'size_adjusted_power_0.10'),
]:
    plt.figure(figsize=(7.5, 5.0))
    for method_id in METHODS:
        subset = POWER_TABLE[
            POWER_TABLE['method_id'] == method_id
        ].sort_values('alpha_x')
        plt.plot(
            subset['alpha_x'],
            subset[value_column],
            marker='o',
            label=METHOD_LABELS[method_id],
        )
    plt.xlabel('Dependence strength alpha_x')
    plt.ylabel('Size-adjusted power')
    plt.title(f'Heteroskedastic DGP: empirical size {level:.2f}')
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Save final results

In [ ]:
OUTPUT_DIR = Path('heteroskedastic_quantile_v2_results')
OUTPUT_DIR.mkdir(exist_ok=True)

POWER_TABLE.to_csv(OUTPUT_DIR / 'size_adjusted_power_long.csv', index=False)
POWER_COMPARISON_005.to_csv(OUTPUT_DIR / 'size_adjusted_power_at_0.05.csv')
POWER_COMPARISON_010.to_csv(OUTPUT_DIR / 'size_adjusted_power_at_0.10.csv')
CALIBRATION_TABLE.to_csv(OUTPUT_DIR / 'h0_calibration_summary.csv', index=False)
LAMBDA_TUNING_TABLE.to_csv(OUTPUT_DIR / 'lambda_tuning_summary.csv', index=False)
pd.DataFrame([{'selected_lambda': SELECTED_LAMBDA}]).to_csv(
    OUTPUT_DIR / 'selected_lambda.csv', index=False
)

np.savez_compressed(
    OUTPUT_DIR / 'h0_pvalues.npz',
    **{
        method_id: np.asarray(result['pvalues'])
        for method_id, result in H0_RESULTS.items()
    },
)

h1_arrays = {}
for method_id, by_alpha in H1_RESULTS.items():
    for alpha_x, result in by_alpha.items():
        alpha_tag = f'{alpha_x:.2f}'.replace('.', 'p')
        h1_arrays[f'{method_id}_alpha_{alpha_tag}'] = np.asarray(result['pvalues'])
np.savez_compressed(OUTPUT_DIR / 'h1_pvalues.npz', **h1_arrays)

print('Selected lambda:', SELECTED_LAMBDA)
print('Saved final results to:', OUTPUT_DIR.resolve())

## Interpretation rule

- Do not select lambda from H1 power; that would tune on the reported outcome.
- The selected lambda is acceptable only if the fresh final H0 results also lie in normal
  Monte-Carlo fluctuation around 0.10 and 0.05.
- If fresh H0 is still liberal, improve generator convergence first.  Do not reuse the
  lambda-tuning p-values as final H0 calibration data.
- Lambda 0.10 is the recommended fallback if a quick code check is needed before the full
  tuning grid is completed.